# Psychophysical kernels (labdata)

Reads fitted `PsychophysicalKernel` rows for an analysis set. CLI alternative:

```bash
uv run python scripts/analyses/plot_psychophysical_kernels.py --analysis-set-id <id> --output figures/kernels.pdf
```

Kernel math lives in `behavior_analyses.kernels`; archived exploratory logic is under `archive/djchurchland/psychophysical_kernels/`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

REPO_ROOT = (
    Path.cwd().parent if Path.cwd().name == "psychophysical_kernels" else Path.cwd()
)
for path in [REPO_ROOT, REPO_ROOT / "src"]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from labdata_plugin.analysisschema import PsychophysicalKernel

ANALYSIS_SET_ID = "example_analysis_set"  # replace after seeding
rows = (
    PsychophysicalKernel() & {"analysis_set_id": ANALYSIS_SET_ID, "fit_status": "fit"}
).fetch(as_dict=True)
assert rows, f"No fitted kernels for {ANALYSIS_SET_ID}"

fig, ax = plt.subplots(figsize=(5, 4))
for row in rows:
    weights_mean = np.asarray(row["weights_mean"], dtype=float)
    weights_error = np.asarray(row["weights_error"], dtype=float)
    x = range(len(weights_mean))
    label = f"{row['subject_name']} (n={row['n_trials_fit']})"
    ax.plot(x, weights_mean, label=label)
    ax.fill_between(
        x, weights_mean - weights_error, weights_mean + weights_error, alpha=0.2
    )
ax.axhline(0, color="k", alpha=0.3, linestyle="--")
ax.set_xlabel("Stimulus time bin")
ax.set_ylabel("Choice weight")
ax.legend(frameon=False, fontsize=8)
ax.set_title("Psychophysical kernels by subject")
fig.show()